In [2]:
# Cell 1: Install required libraries and check GPU
!pip install transformers datasets evaluate accelerate gradio -q

import torch
print(f"✅ GPU available: {torch.cuda.is_available()} - {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

✅ GPU available: True - Tesla T4


In [5]:
# Cell 2: Direct download of IMDB dataset
import os
import tarfile
import urllib.request
from pathlib import Path

# Download IMDB dataset (if not already downloaded)
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
file_path = "aclImdb_v1.tar.gz"

if not os.path.exists(file_path):
    print("Downloading IMDB dataset (84 MB)...")
    urllib.request.urlretrieve(url, file_path)
    print("Download complete!")

# Extract
if not os.path.exists("aclImdb"):
    print("Extracting...")
    with tarfile.open(file_path, "r:gz") as tar:
        tar.extractall()
    print("Extraction complete!")

print("✅ Dataset ready at ./aclImdb/")

Download complete!
Extracting...


/tmp/ipykernel_1666/1260960052.py:20: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


Extraction complete!
✅ Dataset ready at ./aclImdb/


In [6]:
# Cell 3: Read IMDB files and create DataFrame
import pandas as pd
import os

def load_imdb_data(path="aclImdb"):
    # Load positive and negative reviews from train folder
    train_pos = []
    train_neg = []

    # Positive reviews (label=1)
    pos_dir = os.path.join(path, "train", "pos")
    for fname in os.listdir(pos_dir):
        with open(os.path.join(pos_dir, fname), "r", encoding="utf-8") as f:
            train_pos.append(f.read())

    # Negative reviews (label=0)
    neg_dir = os.path.join(path, "train", "neg")
    for fname in os.listdir(neg_dir):
        with open(os.path.join(neg_dir, fname), "r", encoding="utf-8") as f:
            train_neg.append(f.read())

    # Create DataFrame
    train_texts = train_pos + train_neg
    train_labels = [1]*len(train_pos) + [0]*len(train_neg)

    df_train = pd.DataFrame({"text": train_texts, "label": train_labels})
    df_train = df_train.sample(frac=1).reset_index(drop=True)  # Shuffle

    print(f"✅ Training set: {len(df_train)} reviews")
    print(f"   Positive: {len(train_pos)}, Negative: {len(train_neg)}")
    print(f"\nSample:\n{df_train['text'].iloc[0][:200]}...")
    print(f"Label: {df_train['label'].iloc[0]}")

    return df_train

df_train = load_imdb_data()

✅ Training set: 25000 reviews
   Positive: 12500, Negative: 12500

Sample:
Standard "paint-by-numbers" monster fare, filled with a bunch of routine plot devices from big-creature movies. It's like somebody had a deck of cards with plot ideas from other movies written on them...
Label: 0


In [7]:
# Cell 4: Load tokenizer and model
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model with 2 labels (positive/negative)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"✅ Tokenizer loaded: {model_name}")
print(f"✅ Model loaded on: {device}")
print(f"📊 Model parameters: {model.num_parameters():,}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Tokenizer loaded: distilbert-base-uncased
✅ Model loaded on: cuda
📊 Model parameters: 66,955,010


In [9]:
# Cell 5: Tokenize all reviews
from transformers import AutoTokenizer
import torch

# Convert texts to tokenized format
def tokenize_function(texts):
    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

print("Tokenizing 25,000 reviews... This may take 1-2 minutes.")
tokens = tokenize_function(df_train["text"].tolist())

# Convert labels to tensor
labels = torch.tensor(df_train["label"].tolist())

print(f"✅ Tokenization complete. Dataset size: {len(labels)}")
print(f"Sample input_ids shape: {tokens['input_ids'][0].shape[0]}")

Tokenizing 25,000 reviews... This may take 1-2 minutes.
✅ Tokenization complete. Dataset size: 25000
Sample input_ids shape: 512


In [10]:
# Cell 6: Create train/validation splits and DataLoaders
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# Split into train (80%) and validation (20%)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_train["text"].tolist(),
    df_train["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_train["label"].tolist()
)

# Tokenize splits separately
print("Tokenizing train set...")
train_tokens = tokenize_function(train_texts)
print("Tokenizing validation set...")
val_tokens = tokenize_function(val_texts)

# Create TensorDatasets
train_dataset = TensorDataset(
    train_tokens["input_ids"],
    train_tokens["attention_mask"],
    torch.tensor(train_labels)
)

val_dataset = TensorDataset(
    val_tokens["input_ids"],
    val_tokens["attention_mask"],
    torch.tensor(val_labels)
)

# Create DataLoaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"✅ Train samples: {len(train_dataset)}")
print(f"✅ Validation samples: {len(val_dataset)}")
print(f"✅ Batch size: {batch_size}")

Tokenizing train set...
Tokenizing validation set...
✅ Train samples: 20000
✅ Validation samples: 5000
✅ Batch size: 16


In [12]:
# Cell 7: Set up optimizer, loss function, and training loop
from torch.optim import AdamW
from tqdm import tqdm
import torch.nn as nn

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Loss function
loss_fn = nn.CrossEntropyLoss()

# Training settings
num_epochs = 1
print(f"📊 Training for {num_epochs} epoch(s)")
print(f"📈 Learning rate: 2e-5")
print(f"🎯 Setup complete. Ready for training in Cell 8.")

📊 Training for 1 epoch(s)
📈 Learning rate: 2e-5
🎯 Setup complete. Ready for training in Cell 8.


In [13]:
# Cell 8: Training loop (this will take 10-15 minutes)
model.train()
total_steps = len(train_loader) * num_epochs
progress_bar = tqdm(range(total_steps), desc="Training")

for epoch in range(num_epochs):
    total_loss = 0
    for batch_idx, (input_ids, attention_mask, labels) in enumerate(train_loader):
        # Move to GPU
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.update(1)

        # Print progress every 500 batches
        if batch_idx % 500 == 0 and batch_idx > 0:
            avg_loss = total_loss / (batch_idx + 1)
            progress_bar.set_postfix({"loss": f"{avg_loss:.4f}"})

    avg_loss = total_loss / len(train_loader)
    print(f"\n✅ Epoch {epoch+1} completed. Average loss: {avg_loss:.4f}")

print("🎉 Training complete!")

Training: 100%|██████████| 1250/1250 [17:43<00:00,  1.18it/s, loss=0.2656]


✅ Epoch 1 completed. Average loss: 0.2543
🎉 Training complete!


In [14]:
# Cell 9: Evaluate on validation set
from sklearn.metrics import accuracy_score

model.eval()
all_preds = []
all_labels = []

print("Evaluating on validation set...")
with torch.no_grad():
    for input_ids, attention_mask, labels in tqdm(val_loader, desc="Evaluating"):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
print(f"\n✅ Validation Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

Evaluating on validation set...



Evaluating: 100%|██████████| 313/313 [01:31<00:00,  3.41it/s]


✅ Validation Accuracy: 0.9092 (90.92%)


In [15]:
# Cell 10: Build and launch Gradio demo
!pip install gradio -q
import gradio as gr

def predict_sentiment(text):
    # Tokenize input
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)

    # Get prediction
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    sentiment = "Positive 😊" if probs[1] > 0.5 else "Negative 😞"
    confidence = max(probs[0], probs[1])
    return f"{sentiment} (confidence: {confidence:.2%})"

# Create interface
demo = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(lines=3, placeholder="Write a movie review here...", label="Your Review"),
    outputs="text",
    title="🎬 Movie Sentiment Analyzer",
    description="Enter a movie review and the model will predict if it's positive or negative.",
    examples=[
        ["This movie was amazing! Great acting and plot."],
        ["Waste of time. Boring and predictable."],
        ["Not bad, but could have been better."]
    ]
)

# Launch (share=True creates a public link valid for 72 hours)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://469e8cc48c1b749399.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [17]:
!pip install --upgrade huggingface_hub

In [18]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Use the exact same 'model' and 'tokenizer' objects from your training cells
model_repo_name = "your-username/your-model-name"

# Push the model and tokenizer to the Hub
model.push_to_hub(model_repo_name)
tokenizer.push_to_hub(model_repo_name)

print(f"✅ Success! Your model is live at: https://huggingface.co/{model_repo_name}")

In [19]:
# Use your username and a model name
model_repo_name = "mohdali1/distilbert-imdb-sentiment"

# Upload model and tokenizer
model.push_to_hub(model_repo_name)
tokenizer.push_to_hub(model_repo_name)

print(f"✅ Model live at: https://huggingface.co/{model_repo_name}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tyyc893/model.safetensors:   0%|          |  575kB /  268MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

✅ Model live at: https://huggingface.co/mohdali1/distilbert-imdb-sentiment
